# Day 14 - Financial Ratio Validation & Sprint 2 Wrap-up

## Objectives

- Load the Day 12 financial ratios report
- Validate ratio calculations
- Check missing and infinite values
- Identify duplicate company-year records
- Detect financial ratio edge cases
- Generate a final validation report
- Complete Sprint 2 validation

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
df = pd.read_csv("../reports/financial_ratios_day12.csv")

print("Financial Ratios Loaded Successfully")
print("Rows:", len(df))
print("Columns:", len(df.columns))

df.head()

Financial Ratios Loaded Successfully
Rows: 1184
Columns: 7


,company_id,year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,dividend_payout_ratio
0,ABB,Dec 2012,22.411128,0.0,NaN,1.822492,17.241379
1,ABB,Mar 2014,25.126904,0.0,NaN,1.998244,12.626263
2,ABB,Mar 2014,25.126904,0.0,NaN,1.998244,12.626263
3,ABB,Mar 2015,24.439701,0.0,NaN,1.665939,12.663755
4,ABB,Mar 2015,24.439701,0.0,NaN,1.665939,12.663755


In [3]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

Shape: (1184, 7)

Columns:
['company_id', 'year', 'return_on_equity_pct', 'debt_to_equity', 'interest_coverage', 'asset_turnover', 'dividend_payout_ratio']

Data Types:
company_id                   str
year                         str
return_on_equity_pct     float64
debt_to_equity           float64
interest_coverage        float64
asset_turnover           float64
dividend_payout_ratio    float64
dtype: object


In [4]:
missing_report = df.isnull().sum().reset_index()

missing_report.columns = [
    "column_name",
    "missing_values"
]

missing_report["missing_percentage"] = (
    missing_report["missing_values"] / len(df)
) * 100

missing_report

,column_name,missing_values,missing_percentage
0,company_id,0,0.000000
1,year,0,0.000000
2,return_on_equity_pct,0,0.000000
3,debt_to_equity,0,0.000000
4,interest_coverage,93,7.854730
5,asset_turnover,1,0.084459
6,dividend_payout_ratio,4,0.337838


In [5]:
numeric_df = df.select_dtypes(include=[np.number])

infinite_report = pd.DataFrame({
    "column_name": numeric_df.columns,
    "positive_infinity": [
        np.isposinf(numeric_df[col]).sum()
        for col in numeric_df.columns
    ],
    "negative_infinity": [
        np.isneginf(numeric_df[col]).sum()
        for col in numeric_df.columns
    ]
})

infinite_report

,column_name,positive_infinity,negative_infinity
0,return_on_equity_pct,0,0
1,debt_to_equity,0,0
2,interest_coverage,0,0
3,asset_turnover,0,0
4,dividend_payout_ratio,0,0


In [6]:
duplicate_count = df.duplicated(
    subset=["company_id", "year"]
).sum()

print("Duplicate Company-Year Records:", duplicate_count)

Duplicate Company-Year Records: 119


In [7]:
duplicate_records = df[
    df.duplicated(
        subset=["company_id", "year"],
        keep=False
    )
].sort_values(
    by=["company_id", "year"]
)

duplicate_records.head(20)

,company_id,year,return_on_equity_pct,debt_to_equity,interest_coverage,asset_turnover,dividend_payout_ratio
1,ABB,Mar 2014,25.126904,0.000000,NaN,1.998244,12.626263
2,ABB,Mar 2014,25.126904,0.000000,NaN,1.998244,12.626263
3,ABB,Mar 2015,24.439701,0.000000,NaN,1.665939,12.663755
4,ABB,Mar 2015,24.439701,0.000000,NaN,1.665939,12.663755
5,ABB,Mar 2016,21.338912,0.000000,121.666667,1.617574,11.372549
6,ABB,Mar 2016,21.338912,0.000000,121.666667,1.617574,11.372549
7,ABB,Mar 2017,19.971161,0.000000,199.000000,1.405131,11.191336
8,ABB,Mar 2017,19.971161,0.000000,199.000000,1.405131,11.191336
9,ABB,Mar 2018,23.685765,0.000000,131.250000,1.365066,7.231920
10,ABB,Mar 2018,23.685765,0.000000,131.250000,1.365066,7.231920


In [8]:
edge_cases = pd.DataFrame({
    "check": [
        "Negative ROE",
        "ROE Above 100%",
        "Negative Debt to Equity",
        "Debt to Equity Above 5",
        "Negative Interest Coverage",
        "Asset Turnover Above 10",
        "Negative Dividend Payout"
    ],
    "count": [
        (df["return_on_equity_pct"] < 0).sum(),
        (df["return_on_equity_pct"] > 100).sum(),
        (df["debt_to_equity"] < 0).sum(),
        (df["debt_to_equity"] > 5).sum(),
        (df["interest_coverage"] < 0).sum(),
        (df["asset_turnover"] > 10).sum(),
        (df["dividend_payout_ratio"] < 0).sum()
    ]
})

edge_cases

,check,count
0,Negative ROE,73
1,ROE Above 100%,62
2,Negative Debt to Equity,0
3,Debt to Equity Above 5,208
4,Negative Interest Coverage,23
5,Asset Turnover Above 10,41
6,Negative Dividend Payout,4


In [9]:
validation_summary = pd.DataFrame({
    "validation_check": [
        "Total Records",
        "Total Columns",
        "Duplicate Company-Year Records",
        "Columns With Missing Values",
        "Positive Infinity Values",
        "Negative Infinity Values",
        "Edge Case Categories Detected"
    ],
    "result": [
        len(df),
        len(df.columns),
        duplicate_count,
        (df.isnull().sum() > 0).sum(),
        np.isposinf(numeric_df).sum().sum(),
        np.isneginf(numeric_df).sum().sum(),
        (edge_cases["count"] > 0).sum()
    ]
})

validation_summary

,validation_check,result
0,Total Records,1184
1,Total Columns,7
2,Duplicate Company-Year Records,119
3,Columns With Missing Values,3
4,Positive Infinity Values,0
5,Negative Infinity Values,0
6,Edge Case Categories Detected,6


In [10]:
missing_report.to_csv(
    "../reports/day14_missing_values_report.csv",
    index=False
)

infinite_report.to_csv(
    "../reports/day14_infinite_values_report.csv",
    index=False
)

edge_cases.to_csv(
    "../reports/day14_ratio_edge_cases.csv",
    index=False
)

validation_summary.to_csv(
    "../reports/day14_validation_summary.csv",
    index=False
)

print("Day 14 Validation Reports Saved Successfully!")

Day 14 Validation Reports Saved Successfully!


# Day 14 Summary

## Financial Ratio Validation Completed

- Loaded the financial ratios dataset.
- Verified dataset structure and data types.
- Checked missing values.
- Checked positive and negative infinity values.
- Validated duplicate company-year records.
- Identified financial ratio edge cases.
- Generated final validation reports.

## Sprint 2 Status

Financial ratio analysis and validation workflow completed.

The validated financial metrics are ready for the Screener and Peer Comparison Engine.